# 07 · Funcionários da Vulto como autores: raio-x

Fontes: `banco_funcionarios.csv` (o cadastro do `vulto.db`, sem a senha), `banco_vendas.csv`, `batel_vendas_manuais.csv`, `banco_entradas_estoque.csv`, `atendimentos.csv` e `clientes_resgates.csv`.

Objetivo: ver se quem vendeu, lançou o estoque, atendeu ou registrou o resgate chega a uma pessoa do cadastro, para a decisão P9. Na plataforma, a mensagem, o solicitante da transferência e o cancelamento por `equipe` exigem um usuário; o vendedor do pedido e o usuário da movimentação de estoque são opcionais.

> Antes de commitar: *Kernel → Restart & Clear Output* (as tabelas de resgate têm nome e CPF de clientes).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# raiz do repositório, de onde quer que o notebook tenha sido aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "migracao_vulto").is_dir())
STAGING = RAIZ / "staging_vulto"
sys.path.insert(0, str(RAIZ))

from migracao_vulto.normalizadores import chave_busca

# mostra todas as colunas nas tabelas
pd.set_option("display.max_columns", None)

## 1. Carregar

In [ ]:
funcionarios = pd.read_csv(STAGING / "banco_funcionarios.csv", dtype=str)
vendas = pd.read_csv(STAGING / "banco_vendas.csv", dtype=str)
batel = pd.read_csv(STAGING / "batel_vendas_manuais.csv", dtype=str)
entradas = pd.read_csv(STAGING / "banco_entradas_estoque.csv", dtype=str)
atendimentos = pd.read_csv(STAGING / "atendimentos.csv", dtype=str)
resgates = pd.read_csv(STAGING / "clientes_resgates.csv", dtype=str)

## 2. Quem está no cadastro?

Cada linha é uma conta do sistema da Vulto: 24, todas com `ativo = 1` menos `admin`. Nem toda conta é uma pessoa.

In [ ]:
funcionarios[["id", "codigo", "nome", "loja_id", "perfil", "ativo"]]

- **Pessoas (19):** 13 vendedores (Beatriz, Rafael, Rafaela, Camila, Jéssica, Thiago, Larissa, Carlos, Ana Paula, Bianca, Fernanda, Gustavo, Patrícia), 3 gerentes (Marcos, Renata, Hugo) e 3 da sede (Diego, Paula, Marina).
- **Contas genéricas (5):** `gerente batel`, `cd.cajamar`, `TESTE TI`, `Estagiário loja` e `admin`.
- Rafael (`0102`) e Rafaela (`0102`) têm o mesmo código: um sucedeu o outro (seção 3).

## 3. O vendedor da venda chega a uma pessoa?

`vendedor` é texto livre: 48 grafias. Cada grafia aparece numa loja só (fora `Bia`), então a loja diz de quem é o apelido.

In [ ]:
pd.crosstab(vendas["vendedor"].fillna("(vazio)"), vendas["loja_id"])

De-para pela grafia (sem acento e minúscula) **e pela loja**. `Bia` é a Beatriz em Pinheiros (1) e a Bianca na Savassi (3). `Gerente` só aparece na Batel, onde o gerente é a conta genérica `gerente batel`.

In [ ]:
# loja -> grafia (chave_busca) -> id do funcionário
DE_PARA_VENDEDOR = {
    "1": {"beatriz": "1", "beatriz s.": "1", "bia": "1", "bia.pinheiros": "1",
          "rafael": "2", "rafael l.": "2", "rafaela": "3", "rafaela c.": "3",
          "camila": "4", "camila d.": "4", "cami": "4"},
    "2": {"jessica": "6", "jessica p.": "6", "jess.eldorado": "6",
          "thiago": "7", "thiago m": "7", "tiago": "7",
          "larissa": "8", "larissa r.": "8", "lari": "8"},
    "3": {"carlos": "10", "carlinhos": "10", "carlos.savassi": "10",
          "ana paula": "11", "ap": "11", "ana p.": "11", "anapaula": "11",
          "bianca": "12", "bianca t.": "12", "bia": "12"},
    "4": {"fernanda": "14", "fernanda k.": "14", "fer": "14", "fer.batel": "14",
          "gustavo": "15", "gus": "15", "gustavo.batel": "15",
          "patricia": "16", "patricia w.": "16", "pati": "16",
          "gerente": "17", "gerente batel": "17"},
}
APELIDO_RAFA = {"rafa", "rafa.pin"}  # Rafael ou Rafaela: decide pela data (abaixo)

vendas["chave"] = vendas["vendedor"].map(chave_busca)
vendas["data"] = pd.to_datetime(vendas["criado_em"], utc=True)
vendas["id_funcionario"] = [DE_PARA_VENDEDOR.get(loja, {}).get(chave)
                            for loja, chave in zip(vendas["loja_id"], vendas["chave"])]

com_vendedor = vendas["vendedor"].notna()
print("com vendedor:", com_vendedor.sum(), "| sem vendedor (todas online):",
      (~com_vendedor).sum(), vendas.loc[~com_vendedor, "loja_id"].unique())
print("grafias fora do de-para:", sorted(set(vendas.loc[com_vendedor & vendas["id_funcionario"].isna(), "chave"])))

### "Rafa": Rafael ou Rafaela?

Os dois são de Pinheiros e têm o código `0102`. Pelos nomes escritos por extenso, um sucedeu o outro:

In [ ]:
nome_cheio = vendas[vendas["id_funcionario"].isin(["2", "3"])]
nome_cheio.groupby("id_funcionario")["data"].agg(["min", "max", "count"])

In [ ]:
fim_rafael = vendas.loc[vendas["id_funcionario"] == "2", "data"].max()
inicio_rafaela = vendas.loc[vendas["id_funcionario"] == "3", "data"].min()

# até o dia da última venda do Rafael e a partir do dia da primeira da Rafaela; entre os dois fica sem vendedor
rafa = vendas["chave"].isin(APELIDO_RAFA)
dia = vendas["data"].dt.date
vendas.loc[rafa & (dia <= fim_rafael.date()), "id_funcionario"] = "2"
vendas.loc[rafa & (dia >= inicio_rafaela.date()), "id_funcionario"] = "3"

print("Rafa:", rafa.sum(), "| Rafael:", (rafa & (vendas["id_funcionario"] == "2")).sum(),
      "| Rafaela:", (rafa & (vendas["id_funcionario"] == "3")).sum(),
      "| entre os dois (sem vendedor):", (rafa & vendas["id_funcionario"].isna()).sum())
vendas.loc[rafa & vendas["id_funcionario"].isna(), ["id", "criado_em", "vendedor"]]

### Resultado das vendas físicas

In [ ]:
generica = vendas["id_funcionario"] == "17"
pessoa = vendas["id_funcionario"].notna() & ~generica
print("vendas físicas com vendedor:", com_vendedor.sum())
print("  chegam a um dos 13 vendedores:", pessoa.sum(), f"({pessoa.sum() / com_vendedor.sum():.2%})")
print("  conta genérica gerente batel:", generica.sum())
print("  Rafa entre a saída do Rafael e a entrada da Rafaela:", (com_vendedor & vendas["id_funcionario"].isna()).sum())
vendas.loc[pessoa, "id_funcionario"].map(funcionarios.set_index("id")["nome"]).value_counts()

### Contas dos 13 vendedores

E-mail fictício no padrão da demo da Casa Lorenzi (`nome.sobrenome@casalorenzi.example`), trocando só o domínio. `.example` é reservado: não existe nem recebe e-mail.

In [ ]:
DOMINIO_VULTO = "vulto.example"

vendedores = funcionarios[funcionarios["id"].isin(vendas.loc[pessoa, "id_funcionario"].unique())].copy()
vendedores["email"] = vendedores["nome"].map(lambda nome: chave_busca(nome).replace(" ", ".") + "@" + DOMINIO_VULTO)
print("e-mails únicos:", vendedores["email"].is_unique, "| conta da migração:", f"migracao@{DOMINIO_VULTO}")
vendedores[["id", "nome", "loja_id", "email"]]

## 4. Vendas manuais da Batel

Mesma loja (4) e as mesmas grafias. As 4 linhas sem vendedora são os subtotais da planilha (`TOTAL SEMANA 1`… `TOTAL MARÇO`), que saem na V9.

In [ ]:
batel["id_funcionario"] = batel["VENDEDORA"].map(chave_busca).map(DE_PARA_VENDEDOR["4"])
print("linhas:", len(batel), "| com vendedora:", batel["VENDEDORA"].notna().sum(),
      "| chegam a uma pessoa:", batel["id_funcionario"].notna().sum())
pd.crosstab(batel["VENDEDORA"], batel["id_funcionario"].map(funcionarios.set_index("id")["nome"]))

## 5. Quem lançou o estoque?

`entradas_estoque.usuario` é o login usado para lançar, não a pessoa.

In [ ]:
pd.crosstab(entradas["usuario"], entradas["loja_id"], margins=True)

- `pinheiros.loja`, `eldorado.loja`, `savassi.loja`, `batel.loja`: login compartilhado da loja (cada um só na sua loja). `cd.cajamar`: a conta genérica do CD no cadastro.
- `marina.compras`: a Marina (Compras), da sede. `joao.fiscal`: **não está no cadastro**.
- Só 327 linhas de 17.331 têm o nome de uma pessoa do cadastro.

## 6. Quem atendeu?

In [ ]:
nomes_cadastro = set(funcionarios["nome"].map(chave_busca))
primeiros_nomes = set(funcionarios["nome"].map(chave_busca).str.split().str[0])
resumo = atendimentos.groupby("atendente").size().rename("atendimentos").to_frame()
resumo["nome no cadastro"] = [chave_busca(a) in nomes_cadastro for a in resumo.index]
resumo["só o primeiro nome bate"] = [chave_busca(a).replace(".", " ").split()[0] in primeiros_nomes for a in resumo.index]
resumo

In [ ]:
# o atendente aparece em todos os canais na mesma proporção: é a equipe do SAC, não a loja
pd.crosstab(atendimentos["atendente"], atendimentos["canal"])

- Nenhum atendente tem o nome do cadastro. `Marcos SAC` e `Renata`/`renata.sac` têm o primeiro nome de dois gerentes de loja (Marcos Tavares, Pinheiros; Renata Campos, Eldorado), mas assinam como SAC e atendem todos os canais; a Júlia não está no cadastro, e `bot`/`Bot Vulto`/`time insta` não são pessoas.
- Como na A5, o nome sozinho não liga a uma pessoa.

## 7. Quem registrou o resgate?

In [ ]:
com_atendente = resgates[resgates["atendente"].notna()].copy()
com_atendente["chave"] = com_atendente["atendente"].map(chave_busca)
# a loja do resgate é a loja do vendedor? (fora "bia", que é de duas lojas, e o gerente, que toda loja tem)
loja_do_vendedor = {chave: loja for loja, grafias in DE_PARA_VENDEDOR.items() for chave in grafias
                    if chave not in {"bia", "gerente", "gerente batel"}}
NOME_LOJA = {"1": "Pinheiros", "2": "Eldorado", "3": "Savassi", "4": "Batel"}
com_atendente["loja do vendedor"] = com_atendente["chave"].map(loja_do_vendedor).map(NOME_LOJA)
fora = com_atendente["loja do vendedor"].notna() & (com_atendente["loja do vendedor"] != com_atendente["loja"])
print("resgates:", len(resgates), "| com atendente:", len(com_atendente),
      "| em loja que não é a do vendedor:", fora.sum())
pd.crosstab(com_atendente.loc[fora, "atendente"], com_atendente.loc[fora, "loja"])

## Anotações para o dicionário de dados

**O que é cada linha**
- `funcionarios`: uma conta do sistema da Vulto (24). 19 são pessoas e 5 são contas genéricas (`gerente batel`, `cd.cajamar`, `TESTE TI`, `Estagiário loja`, `admin`).

**Autoria em cada fonte**

| Fonte | Campo | Na plataforma | O que é | Chega a uma pessoa? |
|---|---|---|---|---|
| `vendas` | `vendedor` | `pedido.id_registrado_por` (opcional) | apelido do vendedor, 48 grafias | **sim**, pela grafia e pela loja; o "Rafa" pela data |
| Batel | `VENDEDORA` | `pedido.id_registrado_por` | as mesmas grafias da loja 4 | sim, as 72 vendas (as 4 linhas vazias são subtotais) |
| `entradas_estoque` | `usuario` | `movimentacao_estoque.id_usuario` (opcional), `transferencia.id_solicitante` (obrigatório) | login da loja ou do CD | não (só `marina.compras`, 327) |
| `atendimentos` | `atendente` | `mensagem.id_autor` (obrigatório), `chamado.id_responsavel` (opcional) | equipe do SAC e bot | não |
| `Resgates` | `atendente` | movimento de pontos (P5) | apelido do vendedor | não confirma: aparece fora da loja dele |

**Vendas físicas** (10.885 com vendedor; as 7.713 sem vendedor são todas online)
- **10.856 (99,7%) chegam a um dos 13 vendedores.** O "Rafa"/`rafa.pin` (562, Pinheiros) é o Rafael até a última venda dele (30/10/2025) e a Rafaela a partir da primeira dela (11/11/2025); sobram **3 vendas** entre os dois.
- 26 são da conta genérica `gerente batel` (`Gerente`, `gerente batel`, `GERENTE BATEL`, só na Batel).
- O cadastro diz que o Rafael está ativo, mas ele não vende desde out/2025: o `ativo` não é confiável.
- Contas: `nome.sobrenome@vulto.example` (fictício, como o `casalorenzi.example` da demo), todos os nomes separados por ponto e sem acento (`ana.paula.reis`, `jessica.prado`); a conta da migração é `migracao@vulto.example`.

**Resgates** (139 com atendente, de 596)
- 20 estão em loja que não é a do vendedor: `Carlos` (Savassi) em 14 da Batel, do Eldorado e de Pinheiros; `jessica` (Eldorado) em 6. A grafia não diz de quem é o resgate.